# Desafio Gerdau – Diagnóstico da base de códigos de serviço

Notebook com o diagnóstico (EDA) da base anonimizada. Rode as células de cima para baixo.

### Imports

In [ ]:
%pip install pandas

In [73]:
import math
import re
import unicodedata
from collections import Counter

import numpy as np
import pandas as pd

pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 80)

## 1. Carregando e conhecendo a base

In [74]:
ARQUIVO = './content/dataset_gerdau.xlsx'  
ABA = 0                        # 0 = primeira aba (Base Anonimizada)

df = pd.read_excel(ARQUIVO, sheet_name=ABA)
df = df.iloc[:, :5]            # garante só as 5 colunas originais
df.columns = ['grupo', 'fornecedor', 'codigo', 'descricao', 'um']

print(df.shape)
print(df.dtypes)
df.head(10)

(9795, 5)
grupo           str
fornecedor    int64
codigo        int64
descricao       str
um              str
dtype: object


,grupo,fornecedor,codigo,descricao,um
0,Civil,200982674,4033776,SV-EQUIPE TELHADO,SV
1,Civil,201464634,4012475,DIARIA RESGATISTA NIVEL 3 ATE 8H .,SV
2,Civil,200772986,4050934,SV OFICIAL MENSAL,SV
3,Civil,300623388,4097173,Pavimentação a frio por kg,KG
4,Civil,200982674,4066840,SV-EQUIPE PINTURA,SV
5,Civil,200772986,4050928,SV ENCARREGADO MENSAL,SV
6,Civil,201332350,4096183,ELETRICISTA - ROTINA,SV
7,Civil,200772986,4050925,SV ENGENHEIRO MENSAL,SV
8,Civil,300056198,4023627,SV-HORA OFICIAL-HR NORMAL,H
9,Civil,201464634,4017177,Diaria resgatista turno de ate 8 horas .,SV


## 2. Tamanho e qualidade

Cada linha é a relação **código de serviço × fornecedor**, então linhas ≠ códigos distintos.

In [75]:
print('Linhas:', len(df))
print('Códigos de serviço distintos:', df['codigo'].nunique())
print('Fornecedores distintos:', df['fornecedor'].nunique())
print('\nNulos por coluna:')
print(df.isna().sum())

Linhas: 9795
Códigos de serviço distintos: 7937
Fornecedores distintos: 111

Nulos por coluna:
grupo         0
fornecedor    0
codigo        0
descricao     0
um            0
dtype: int64


In [76]:
resumo_grupo = df.groupby('grupo').agg(
    linhas=('codigo', 'size'),
    codigos=('codigo', 'nunique'),
    fornecedores=('fornecedor', 'nunique'))
resumo_grupo

,linhas,codigos,fornecedores
grupo,,,
Civil,4558,3451,24
Eletromecânica,2456,1779,29
Facilities,2781,2730,62


In [77]:
print('Unidades de medida:')
print(df['um'].value_counts())

Unidades de medida:
um
SV     4083
H      3152
ST     1039
M2      571
M       444
M3      159
MON     139
KG       71
CM       47
TAG      37
ZCJ      16
KM       15
L         8
CDM       6
MM        4
TO        2
ML        2
Name: count, dtype: int64


## 3. Normalização

In [78]:
def norm(s):
    """Maiúsculas, sem acento, sem pontuação e sem espaços extras."""

    s = (
        unicodedata.normalize('NFKD', str(s))
        .encode('ascii', 'ignore')
        .decode()
        .upper()
    )

    s = s.replace('/', ' ')
    s = s.replace('+', ' + ')

    s = re.sub(r'[-_.,;:()"]+', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()

    return s


# Descrição original em maiúsculas
df['descricao_lit'] = (
    df['descricao']
    .astype(str)
    .str.upper()
    .str.strip()
)

# Descrição normalizada
df['descricao_norm'] = df['descricao'].map(norm)

# Conferir 10 exemplos
df[['descricao', 'descricao_lit', 'descricao_norm']].sample(
    10,
    random_state=1
)


,descricao,descricao_lit,descricao_norm
2573,LÍDER ELÉTRICA - HORA INSALUBRE CHEIA,LÍDER ELÉTRICA - HORA INSALUBRE CHEIA,LIDER ELETRICA HORA INSALUBRE CHEIA
7632,SV-AUXILIAR SERVIÇOS GERAIS-ADM,SV-AUXILIAR SERVIÇOS GERAIS-ADM,SV AUXILIAR SERVICOS GERAIS ADM
1755,SV VIDRO LISO 3 MILÍMETROS,SV VIDRO LISO 3 MILÍMETROS,SV VIDRO LISO 3 MILIMETROS
7759,H.E.SS-Eletricista NR35-ADM,H.E.SS-ELETRICISTA NR35-ADM,H E SS ELETRICISTA NR35 ADM
6964,MONTADOR(A) LIDER(HE NOT DOM.FER),MONTADOR(A) LIDER(HE NOT DOM.FER),MONTADOR A LIDER HE NOT DOM FER
5701,PREVENTIVA PONTES - 2415 - USB1,PREVENTIVA PONTES - 2415 - USB1,PREVENTIVA PONTES 2415 USB1
5880,SOLDADOR MAÇARIQUEIRO (HORA NOT),SOLDADOR MAÇARIQUEIRO (HORA NOT),SOLDADOR MACARIQUEIRO HORA NOT
7652,VIGILANTE DESARMADO 24H (SEG a DOM),VIGILANTE DESARMADO 24H (SEG A DOM),VIGILANTE DESARMADO 24H SEG A DOM
7834,H.E.SS-Sup.Áreas Verdes+mot.caminhão-ADM,H.E.SS-SUP.ÁREAS VERDES+MOT.CAMINHÃO-ADM,H E SS SUP AREAS VERDES + MOT CAMINHAO ADM
234,SV-OFICIAL COM MARTELO ROMPEDOR OU ELÉTR,SV-OFICIAL COM MARTELO ROMPEDOR OU ELÉTR,SV OFICIAL COM MARTELO ROMPEDOR OU ELETR


In [79]:
# Só compara descrições do MESMO grupo de serviço e da MESMA unidade de medida.

def tokens_palavra(texto):
    # Equivale ao TfidfVectorizer(analyzer='word', ngram_range=(1, 2))
    p = re.findall(r'(?u)\b\w\w+\b', texto.lower())
    return p + [f'{a} {b}' for a, b in zip(p, p[1:])]

def tokens_char_wb(texto, n_min=3, n_max=5):
    # Equivale ao TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5))
    out = []
    for w in texto.lower().split():
        w = f' {w} '
        for n in range(n_min, n_max + 1):
            off = 0
            out.append(w[off:off + n])
            while off + n < len(w):
                off += 1
                out.append(w[off:off + n])
            if off == 0:
                break
    return out

def tfidf(textos, tokenizador):
    """Retorna, para cada texto, um dict {termo: peso} já normalizado (norma L2)."""
    contagens = [Counter(tokenizador(t)) for t in textos]
    n = len(textos)
    df_termo = Counter(t for c in contagens for t in c)
    idf = {t: math.log((1 + n) / (1 + d)) + 1 for t, d in df_termo.items()}
    vetores = []
    for c in contagens:
        v = {t: q * idf[t] for t, q in c.items()}
        norma = math.sqrt(sum(x * x for x in v.values())) or 1.0
        vetores.append({t: x / norma for t, x in v.items()})
    return vetores

def matriz_densa(vetores, idx):
    cols = {}
    for i in idx:
        for t in vetores[i]:
            cols.setdefault(t, len(cols))
    M = np.zeros((len(idx), len(cols)), dtype=np.float32)
    for r, i in enumerate(idx):
        for t, x in vetores[i].items():
            M[r, cols[t]] = x
    return M

def componentes(n, pares):
    pai = list(range(n))
    def raiz(x):
        while pai[x] != x:
            pai[x] = pai[pai[x]]
            x = pai[x]
        return x
    for a, b in pares:
        ra, rb = raiz(a), raiz(b)
        if ra != rb:
            pai[ra] = rb
    return [raiz(i) for i in range(n)]


LIMITE_SIMILARIDADE = 0.72
PESO_PALAVRA = 0.60          # peso do TF-IDF de palavras; o de caracteres é 1 - PESO_PALAVRA

colunas_chave = ['descricao_norm', 'grupo', 'um']
chaves = df[colunas_chave].drop_duplicates().reset_index(drop=True)

textos = chaves['descricao_norm'].tolist()
vet_palavra = tfidf(textos, tokens_palavra)
vet_char = tfidf(textos, tokens_char_wb)

chaves['grupo_semantico'] = 0
proximo_id = 1

for _, idx in chaves.groupby(['grupo', 'um']).indices.items():
    if len(idx) < 2:
        continue
    P = matriz_densa(vet_palavra, idx)
    C = matriz_densa(vet_char, idx)
    sim = PESO_PALAVRA * (P @ P.T) + (1 - PESO_PALAVRA) * (C @ C.T)
    np.fill_diagonal(sim, 0)

    a, b = np.where(np.triu(sim >= LIMITE_SIMILARIDADE, 1))
    raiz = componentes(len(idx), zip(a, b))
    tamanho = Counter(raiz)

    ids = {}
    for k, r in enumerate(raiz):
        if tamanho[r] >= 2:                      # só famílias com pelo menos 2 descrições
            if r not in ids:
                ids[r] = proximo_id
                proximo_id += 1
            chaves.loc[idx[k], 'grupo_semantico'] = ids[r]

df = df.drop(columns='grupo_semantico', errors='ignore').merge(chaves, on=colunas_chave, how='left')
df['grupo_semantico'] = df['grupo_semantico'].fillna(0).astype(int)
df.shape

(9795, 8)

In [80]:
# Resumo do diagnóstico
candidatas = df[df['grupo_semantico'] > 0]

resumo = pd.DataFrame([
    ('Linhas totais',
     len(df),
     'Todas as linhas foram preservadas.'),
    ('Descrições brutas distintas',
     df['descricao'].nunique(),
     'Quantidade de textos diferentes antes da normalização.'),
    ('Descrições normalizadas distintas',
     df['descricao_norm'].nunique(),
     'Após caixa alta, remoção de acentos e pontuação.'),
    ('Chaves descrição+grupo+unidade distintas',
     df[['descricao_norm', 'grupo', 'um']].drop_duplicates().shape[0],
     'Evita misturar registros de grupos/unidades diferentes.'),
    ('Linhas em duplicidade exata após normalização',
     int(df.duplicated(subset=['descricao_norm', 'grupo', 'um'], keep=False).sum()),
     'Candidatas a consolidação determinística antes de IA.'),
    ('Descrições distintas em grupos semânticos candidatos',
     candidatas['descricao_norm'].nunique(),
     'Agrupamentos com pelo menos 2 descrições.'),
    ('Linhas pertencentes a grupos candidatos',
     len(candidatas),
     'Linhas que receberam Grupo Semântico > 0.'),
    ('Grupos semânticos candidatos',
     candidatas['grupo_semantico'].nunique(),
     'Famílias geradas pelo método híbrido TF-IDF palavra+caractere.'),
    ('Limite de similaridade para unir',
     LIMITE_SIMILARIDADE,
     'Somente mesmo Grupo de Serviço + mesma Unidade de medida.'),
], columns=['Métrica', 'Valor', 'Observação'], dtype=object)

# Inteiros sem casas decimais; o limite com vírgula decimal (0,72)
resumo['Valor'] = [
    f'{v:.2f}'.replace('.', ',') if isinstance(v, float) else str(int(v))
    for v in resumo['Valor']
]
display(resumo)

# Exportar no mesmo formato da tabela de referência
# resumo.to_excel('resumo_diagnostico.xlsx', index=False, header=False)

,Métrica,Valor,Observação
0,Linhas totais,9795,Todas as linhas foram preservadas.
1,Descrições brutas distintas,7468,Quantidade de textos diferentes antes da normalização.
2,Descrições normalizadas distintas,7333,"Após caixa alta, remoção de acentos e pontuação."
3,Chaves descrição+grupo+unidade distintas,7397,Evita misturar registros de grupos/unidades diferentes.
4,Linhas em duplicidade exata após normalização,3885,Candidatas a consolidação determinística antes de IA.
5,Descrições distintas em grupos semânticos candidatos,4021,Agrupamentos com pelo menos 2 descrições.
6,Linhas pertencentes a grupos candidatos,5721,Linhas que receberam Grupo Semântico > 0.
7,Grupos semânticos candidatos,1119,Famílias geradas pelo método híbrido TF-IDF palavra+caractere.
8,Limite de similaridade para unir,"0,72",Somente mesmo Grupo de Serviço + mesma Unidade de medida.


In [81]:

# 1. Linhas totais
linhas_totais = len(df)

# 2. Descrições brutas distintas
descricoes_brutas = df['descricao'].nunique()

# 3. Descrições normalizadas distintas
descricoes_norm = df['descricao_norm'].nunique()

# 4. Chaves distintas: descrição + grupo + unidade
colunas_chave = ['descricao_norm', 'grupo', 'um']

chaves_distintas = df[colunas_chave].drop_duplicates().shape[0]

# 5. Linhas que fazem parte de duplicações
duplicadas = df.duplicated(
    subset=colunas_chave,
    keep=False
)

linhas_duplicadas = int(duplicadas.sum())

# Exibir resultados
print('Linhas totais:', linhas_totais)
print('Descrições brutas distintas:', descricoes_brutas)
print('Descrições normalizadas distintas:', descricoes_norm)
print('Chaves distintas:', chaves_distintas)
print('Linhas duplicadas:', linhas_duplicadas)


Linhas totais: 9795
Descrições brutas distintas: 7468
Descrições normalizadas distintas: 7333
Chaves distintas: 7397
Linhas duplicadas: 3885


In [82]:
def norm_teste(s):
    s = unicodedata.normalize('NFKD', str(s))
    s = s.encode('ascii', 'ignore').decode().upper()

    # Preserva o sinal +, mas separa barras
    s = s.replace('/', ' ')
    s = s.replace('+', ' PLUS ')

    # Remove toda pontuação restante
    s = re.sub(r'[^A-Z0-9\s]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()

    # Restaura o sinal + representado por PLUS
    s = s.replace(' PLUS ', ' + ')

    return s

# Comparar as duas normalizações
comparacao = pd.DataFrame({
    'original': df['descricao'],
    'norm_atual': df['descricao_norm'],
    'norm_teste': df['descricao'].map(norm_teste)
})

print("Descrições distintas na normalização atual:",
      comparacao['norm_atual'].nunique())

print("Descrições distintas no teste:",
      comparacao['norm_teste'].nunique())

# Mostrar exemplos que mudaram
mudancas = comparacao[
    comparacao['norm_atual'] != comparacao['norm_teste']
]

display(mudancas.drop_duplicates().head(30))


Descrições distintas na normalização atual: 7333
Descrições distintas no teste: 7330


,original,norm_atual,norm_teste
27,"SV-ARMADURA CA-50 D <= 1/2"" MAT FORNECED",SV ARMADURA CA 50 D <= 1 2 MAT FORNECED,SV ARMADURA CA 50 D 1 2 MAT FORNECED
90,SV-MONTADOR TELHADO HE 60%,SV MONTADOR TELHADO HE 60%,SV MONTADOR TELHADO HE 60
103,SV-AJUD TELHADO HE 60%,SV AJUD TELHADO HE 60%,SV AJUD TELHADO HE 60
110,SV-MONTADOR TELHADO HE 100%,SV MONTADOR TELHADO HE 100%,SV MONTADOR TELHADO HE 100
123,SV-AJUD TELHADO HE 100%,SV AJUD TELHADO HE 100%,SV AJUD TELHADO HE 100
160,SV-PINTOR HE 60%,SV PINTOR HE 60%,SV PINTOR HE 60
165,"ARMADURA CA-50 DN >=16,0MM INCLUSIVE CO",ARMADURA CA 50 DN >=16 0MM INCLUSIVE CO,ARMADURA CA 50 DN 16 0MM INCLUSIVE CO
193,SV-ENCARREG PINTURA HE 60%,SV ENCARREG PINTURA HE 60%,SV ENCARREG PINTURA HE 60
209,HIDRÁULICO - HORA EXTRA 50%,HIDRAULICO HORA EXTRA 50%,HIDRAULICO HORA EXTRA 50
217,SV-TEC SEGUR (TELHADO) HE 60%,SV TEC SEGUR TELHADO HE 60%,SV TEC SEGUR TELHADO HE 60


In [83]:
# Buscar descrições que começam com SV, H ou SV/H
padrao = r'^\s*(?:SV\s*/\s*H|SV|H)\b'

prefixos = df[
    df['descricao'].astype(str).str.contains(
        padrao,
        case=False,
        regex=True,
        na=False
    )
]

display(prefixos[['descricao', 'grupo', 'um']].head(30))

print("Linhas com possíveis prefixos:", len(prefixos))


,descricao,grupo,um
0,SV-EQUIPE TELHADO,Civil,SV
2,SV OFICIAL MENSAL,Civil,SV
4,SV-EQUIPE PINTURA,Civil,SV
5,SV ENCARREGADO MENSAL,Civil,SV
7,SV ENGENHEIRO MENSAL,Civil,SV
8,SV-HORA OFICIAL-HR NORMAL,Civil,H
11,SV TEC DE PLANEJAMENTO MENSAL,Civil,SV
14,SV TEC DE SEGURANÇA MENSAL,Civil,SV
17,SV MARTELETEIRO MENSAL,Civil,SV
18,SV AJUDANTE MENSAL,Civil,SV


Linhas com possíveis prefixos: 4687


In [84]:

# Identifica apenas SV/H, SV ou H no início da descrição,
# seguido de espaço, barra ou fim do texto.
padrao = r'^\s*(SV\s*/\s*H|SV|H)(?=\s|/|$)'

prefixos = df[
    df['descricao'].astype(str).str.contains(
        padrao, case=False, regex=True, na=False
    )
].copy()

print("Linhas identificadas:", len(prefixos))
print("\nDistribuição dos prefixos:")

prefixos['prefixo'] = (
    prefixos['descricao']
    .astype(str)
    .str.extract(padrao, flags=re.IGNORECASE)[0]
    .str.upper()
    .str.replace(r'\s+', '', regex=True)
)

print(prefixos['prefixo'].value_counts(dropna=False))

display(prefixos[['descricao', 'descricao_norm', 'grupo', 'um']].head(40))


Linhas identificadas: 2870

Distribuição dos prefixos:
prefixo
SV    2785
H       85
Name: count, dtype: int64


C:\Users\schei\AppData\Local\Temp\ipykernel_4828\1316588474.py:6: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df['descricao'].astype(str).str.contains(


,descricao,descricao_norm,grupo,um
2,SV OFICIAL MENSAL,SV OFICIAL MENSAL,Civil,SV
5,SV ENCARREGADO MENSAL,SV ENCARREGADO MENSAL,Civil,SV
7,SV ENGENHEIRO MENSAL,SV ENGENHEIRO MENSAL,Civil,SV
11,SV TEC DE PLANEJAMENTO MENSAL,SV TEC DE PLANEJAMENTO MENSAL,Civil,SV
14,SV TEC DE SEGURANÇA MENSAL,SV TEC DE SEGURANCA MENSAL,Civil,SV
17,SV MARTELETEIRO MENSAL,SV MARTELETEIRO MENSAL,Civil,SV
18,SV AJUDANTE MENSAL,SV AJUDANTE MENSAL,Civil,SV
19,SV - ESCAVADEIRA HIDRÁULICA 17Ton,SV ESCAVADEIRA HIDRAULICA 17TON,Civil,SV
20,SV RETRO ESCAVADEIRA MENSAL,SV RETRO ESCAVADEIRA MENSAL,Civil,SV
21,SV VEICULO UTILITARIO TIPO VAN - MENSAL,SV VEICULO UTILITARIO TIPO VAN MENSAL,Civil,SV


### Testar a remoção dos prefixos
Execute esta célula para comparar três cenários sem alterar o df original:

In [85]:
def norm_base(s):
    s = unicodedata.normalize('NFKD', str(s))
    s = s.encode('ascii', 'ignore').decode().upper()
    s = s.replace('/', ' ')
    s = s.replace('+', ' + ')
    s = re.sub(r'[-_.,;:()"]+', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def norm_sem_sv_h(s):
    s = norm_base(s)
    s = re.sub(r'^(?:SV\s*/\s*H|SV|H)\b\s*', '', s)
    return re.sub(r'\s+', ' ', s).strip()

def norm_sv_h_unificado(s):
    s = norm_base(s)
    s = re.sub(r'^(?:SV\s*/\s*H|SV|H)\b\s*', 'SV H ', s)
    return re.sub(r'\s+', ' ', s).strip()

teste = pd.DataFrame({
    'atual': df['descricao'].map(norm_base),
    'sem_prefixo': df['descricao'].map(norm_sem_sv_h),
    'prefixo_unificado': df['descricao'].map(norm_sv_h_unificado)
})

print("Atual:", teste['atual'].nunique())
print("Sem prefixo:", teste['sem_prefixo'].nunique())
print("Prefixo unificado:", teste['prefixo_unificado'].nunique())

display(
    pd.DataFrame({
        'original': df['descricao'],
        'atual': teste['atual'],
        'sem_prefixo': teste['sem_prefixo'],
        'prefixo_unificado': teste['prefixo_unificado']
    })
    .loc[teste['atual'] != teste['sem_prefixo']]
    .drop_duplicates()
    .head(30)
)

Atual: 7333
Sem prefixo: 7302
Prefixo unificado: 7333


,original,atual,sem_prefixo,prefixo_unificado
0,SV-EQUIPE TELHADO,SV EQUIPE TELHADO,EQUIPE TELHADO,SV H EQUIPE TELHADO
2,SV OFICIAL MENSAL,SV OFICIAL MENSAL,OFICIAL MENSAL,SV H OFICIAL MENSAL
4,SV-EQUIPE PINTURA,SV EQUIPE PINTURA,EQUIPE PINTURA,SV H EQUIPE PINTURA
5,SV ENCARREGADO MENSAL,SV ENCARREGADO MENSAL,ENCARREGADO MENSAL,SV H ENCARREGADO MENSAL
7,SV ENGENHEIRO MENSAL,SV ENGENHEIRO MENSAL,ENGENHEIRO MENSAL,SV H ENGENHEIRO MENSAL
8,SV-HORA OFICIAL-HR NORMAL,SV HORA OFICIAL HR NORMAL,HORA OFICIAL HR NORMAL,SV H HORA OFICIAL HR NORMAL
11,SV TEC DE PLANEJAMENTO MENSAL,SV TEC DE PLANEJAMENTO MENSAL,TEC DE PLANEJAMENTO MENSAL,SV H TEC DE PLANEJAMENTO MENSAL
14,SV TEC DE SEGURANÇA MENSAL,SV TEC DE SEGURANCA MENSAL,TEC DE SEGURANCA MENSAL,SV H TEC DE SEGURANCA MENSAL
17,SV MARTELETEIRO MENSAL,SV MARTELETEIRO MENSAL,MARTELETEIRO MENSAL,SV H MARTELETEIRO MENSAL
18,SV AJUDANTE MENSAL,SV AJUDANTE MENSAL,AJUDANTE MENSAL,SV H AJUDANTE MENSAL


Execute esta célula para identificar quais descrições passam a ser iguais quando removemos os prefixos:

In [86]:
comparacao_prefixos = pd.DataFrame({
    'original': df['descricao'].astype(str),
    'normalizada': df['descricao'].map(norm_base),
    'sem_prefixo': df['descricao'].map(norm_sem_sv_h),
    'grupo': df['grupo'],
    'um': df['um']
})

# Descrições diferentes que se tornam iguais após remover o prefixo
grupos = (
    comparacao_prefixos
    .groupby('sem_prefixo')
    .agg(
        descricoes_originais=('normalizada', 'nunique'),
        exemplos=('original', lambda x: list(pd.unique(x))[:5]),
        grupos=('grupo', lambda x: list(pd.unique(x))[:5]),
        unidades=('um', lambda x: list(pd.unique(x))[:5])
    )
    .query('descricoes_originais > 1')
    .sort_values('descricoes_originais', ascending=False)
)

display(grupos.head(30))

,descricoes_originais,exemplos,grupos,unidades
sem_prefixo,,,,
CAMINHAO CACAMBA,2,"[Caminhão caçamba, SV-CAMINHAO CACAMBA .]",[Civil],[SV]
COORDENADOR,2,"[Coordenador, SV - Coordenador]",[Facilities],[SV]
ELETRICISTA MENSAL,2,"[SV ELETRICISTA MENSAL, SV - Eletricista- (mensal), Eletricista Mensal]","[Civil, Eletromecânica, Facilities]",[SV]
ENCARREGADO,2,"[ENCARREGADO, SV-ENCARREGADO, SV ENCARREGADO]","[Civil, Eletromecânica, Facilities]","[SV, H]"
ENCARREGADO HORA EXTRA 100%,2,"[SV-ENCARREGADO-HORA EXTRA 100% ., ENCARREGADO - HORA EXTRA 100%]",[Civil],"[SV, H]"
ENCARREGADO MENSAL,2,"[SV ENCARREGADO MENSAL, Encarregado MENSAL]",[Civil],[SV]
GERADOR 10KVA HORA IMPRODUTIVA,2,"[SV GERADOR 10kVA - HORA IMPRODUTIVA, GERADOR 10kVA - HORA IMPRODUTIVA]",[Civil],[H]
GERADOR 10KVA HORA PRODUTIVA,2,"[SV GERADOR 10kVA - HORA PRODUTIVA, GERADOR 10kVA - HORA PRODUTIVA]",[Civil],[H]
GIRO ZERO,2,"[SV - GIRO ZERO, GIRO ZERO]",[Facilities],[SV]


## 4. Repetição

"Repetido" tem três significados. Medimos um de cada vez:

- **4a.** Mesmo código em várias linhas.
- **4b.** Mesma descrição em códigos diferentes.
- **4c.** Mesmo significado com descrições diferentes.

### 4a. Mesmo código em mais de uma linha

In [87]:
por_codigo = df.groupby('codigo').agg(
    linhas=('codigo', 'size'),
    fornecedores=('fornecedor', 'nunique'),
    descricoes=('descricao_norm', 'nunique'),
    grupos=('grupo', 'nunique'),
    unidades=('um', 'nunique'))

varias = por_codigo[por_codigo['linhas'] > 1]

print('Códigos distintos:', len(por_codigo))
print('Códigos em mais de uma linha:', len(varias))
print('Linhas desses códigos:', int(varias['linhas'].sum()))
print('Pares código+fornecedor repetidos:',
      int(df.duplicated(['codigo', 'fornecedor'], keep=False).sum()))

print('\nQuantas linhas cada código tem:')
print(por_codigo['linhas'].value_counts().sort_index())

# Repetir código é esperado (um código × vários fornecedores).
# O problema é o MESMO código com descrição, grupo ou unidade diferentes.
conflitos = por_codigo[
    (por_codigo['descricoes'] > 1) |
    (por_codigo['grupos'] > 1) |
    (por_codigo['unidades'] > 1)
]

print('\nCódigos com descrição diferente:', int((por_codigo['descricoes'] > 1).sum()))
print('Códigos em mais de um grupo de serviço:', int((por_codigo['grupos'] > 1).sum()))
print('Códigos com mais de uma unidade:', int((por_codigo['unidades'] > 1).sum()))

df[df['codigo'].isin(conflitos.index)].sort_values('codigo')[
    ['codigo', 'fornecedor', 'grupo', 'um', 'descricao']
].head(15)

Códigos distintos: 7937
Códigos em mais de uma linha: 1234
Linhas desses códigos: 3092
Pares código+fornecedor repetidos: 0

Quantas linhas cada código tem:
linhas
1    6703
2     916
3      93
4     155
5      59
6      11
Name: count, dtype: int64

Códigos com descrição diferente: 1
Códigos em mais de um grupo de serviço: 23
Códigos com mais de uma unidade: 0


,codigo,fornecedor,grupo,um,descricao
739,4035891,201341488,Civil,SV,SV-CALDEIRARIA(MENSAL DEMANDA )
4574,4035891,200876404,Eletromecânica,SV,SV-CALDEIRARIA(MENSAL DEMANDA )
4575,4035891,300805670,Eletromecânica,SV,SV-CALDEIRARIA(MENSAL DEMANDA )
740,4035892,201341488,Civil,SV,SV-CALDEIRARIA(QUINZENAL DEMANDA)
4577,4035892,300805670,Eletromecânica,SV,SV-CALDEIRARIA(QUINZENAL DEMANDA)
4576,4035892,200876404,Eletromecânica,SV,SV-CALDEIRARIA(QUINZENAL DEMANDA)
741,4035893,201341488,Civil,SV,SV-CALDEIRARIA(SEMANAL DEMANDA)
4578,4035893,200876404,Eletromecânica,SV,SV-CALDEIRARIA(SEMANAL DEMANDA)
4579,4035893,300805670,Eletromecânica,SV,SV-CALDEIRARIA(SEMANAL DEMANDA)
4580,4035894,200876404,Eletromecânica,SV,SV-CALDEIRARIA(DIARIA DEMANDA)


**Como ler:** repetir o código em várias linhas é normal, porque cada linha é a relação código × fornecedor. Nenhum par código+fornecedor aparece duas vezes. O que importa são os **conflitos**: o mesmo código com descrição ou grupo diferentes, que precisam ser corrigidos na origem e não podem ser tratados como duplicidade.

### 4b. Mesma descrição em códigos diferentes

In [88]:
por_desc = df.groupby('descricao_norm').agg(
    codigos=('codigo', 'nunique'),
    grupos=('grupo', 'nunique'),
    unidades=('um', 'nunique'))

repetidas = por_desc[por_desc['codigos'] > 1]

print('Descrições normalizadas distintas:', len(por_desc))
print('Descrições presentes em mais de um código:', len(repetidas))
print('Códigos envolvidos:',
      df[df['descricao_norm'].isin(repetidas.index)]['codigo'].nunique())

mesmo_gu = (repetidas['grupos'] == 1) & (repetidas['unidades'] == 1)
print('\n  mesmo grupo e mesma unidade:', int(mesmo_gu.sum()))
print('  grupos diferentes:', int((repetidas['grupos'] > 1).sum()))
print('  mesmo grupo, unidades diferentes:',
      int(((repetidas['grupos'] == 1) & (repetidas['unidades'] > 1)).sum()))

# Consolidação segura: mesma descrição + grupo + unidade com mais de um código
por_chave = df.groupby(['descricao_norm', 'grupo', 'um'])['codigo'].nunique()
print('\nChaves descrição+grupo+unidade com mais de um código:', int((por_chave > 1).sum()))
print('Códigos excedentes (podem ser unificados):', int((por_chave - 1)[por_chave > 1].sum()))

repetidas.sort_values('codigos', ascending=False).head(10)

Descrições normalizadas distintas: 7333
Descrições presentes em mais de um código: 486
Códigos envolvidos: 1091

  mesmo grupo e mesma unidade: 448
  grupos diferentes: 20
  mesmo grupo, unidades diferentes: 18

Chaves descrição+grupo+unidade com mais de um código: 457
Códigos excedentes (podem ser unificados): 564


,codigos,grupos,unidades
descricao_norm,,,
SV FURACAO EM CONCRETO COM COROA DIAM,13,1,1
SV ANCORAGEM QUIMICA SISTEMA HVA DA HI,13,1,2
SV ANCORAGEM QUIMICA SISTEMA HIT FOR,7,1,1
MES SERVENTE DE OBRAS ADM,6,1,1
H P E DF SERVENTE DE OBRAS ADM,5,1,1
SV PORTA PRANCHETA COMPLETA PARA PINTURA,5,1,1
H I + E DF SERVENTE DE OBRAS ADM,5,1,1
AJUDANTE HORA NORMAL,5,2,1
SV LIMP E CONSERV AUX DE SERVIC HE 100%,4,1,1


**Como ler:** a mesma descrição com grupo e unidade iguais, mas código diferente, é o caso de consolidação mais seguro e **não precisa de IA**. Quando muda o grupo ou a unidade, a descrição igual pode ser serviço diferente, então esses casos ficam para revisão.

### 4c. Mesmo significado, descrições diferentes (candidatos do agrupamento semântico)

In [89]:
candidatas = df[df['grupo_semantico'] > 0]

familias = candidatas.groupby('grupo_semantico').agg(
    descricoes=('descricao_norm', 'nunique'),
    codigos=('codigo', 'nunique'),
    linhas=('codigo', 'size'))

descricoes_cand = candidatas[['grupo_semantico', 'descricao_norm']].drop_duplicates()

print('Grupos semânticos candidatos:', len(familias))
print('Descrições distintas nesses grupos:', descricoes_cand['descricao_norm'].nunique())
print('Descrições excedentes se cada grupo virasse 1:',
      descricoes_cand['descricao_norm'].nunique() - len(familias))
print('Códigos de serviço envolvidos:', candidatas['codigo'].nunique())

print('\nDescrições por grupo:')
print(familias['descricoes'].describe()[['min', '50%', 'max']].astype(int).to_string())

# Sinal de alerta: números diferentes (ex.: 50 x 100, 1/2 x 3/4) costumam ser serviços diferentes
def assinatura_numerica(descricoes):
    return len({tuple(re.findall(r'\d+', d)) for d in descricoes})

dif_num = descricoes_cand.groupby('grupo_semantico')['descricao_norm'].apply(assinatura_numerica)
print('\nGrupos com números diferentes entre as descrições:', int((dif_num > 1).sum()))
print('Grupos sem diferença numérica:', int((dif_num == 1).sum()))

# Amostra de grupos sem diferença numérica (melhores candidatos)
amostra = dif_num[dif_num == 1].sample(5, random_state=1).index
(descricoes_cand[descricoes_cand['grupo_semantico'].isin(amostra)]
 .sort_values(['grupo_semantico', 'descricao_norm']))

Grupos semânticos candidatos: 1119
Descrições distintas nesses grupos: 4021
Descrições excedentes se cada grupo virasse 1: 2902
Códigos de serviço envolvidos: 4434

Descrições por grupo:
min     2
50%     2
max    71

Grupos com números diferentes entre as descrições: 481
Grupos sem diferença numérica: 638


,grupo_semantico,descricao_norm
2519,122,AUXILIAR DE SERVICOS GERAIS HNOTUR DIS
2518,122,AUXILIAR DE SERVICOS GERAIS HNOTUR RES
6245,762,FABR CURVA CARBON COLETOR INF ATE1000MM
6249,762,FABR CURVA INCONEL COLETOR INF ATE1000MM
7342,918,SV TECNICO DE SEGURANCA DO TRABALHO
7781,918,SV TECNICO SEGURANCA DO TRABALHO CSG
8922,918,TECNICO DE SEGURANCA DO TRABALHO
8493,1037,MANUTENCAO PREDIAL
8809,1037,OFICIAL MANUTENCAO PREDIAL
8376,1037,SV MANUTENCAO PREDIAL


**Como ler:** estes são **candidatos**, não duplicidades confirmadas. O método encadeia semelhanças, então grupos grandes podem misturar serviços diferentes, e os que têm números distintos (potências, diâmetros, percentuais) são os mais suspeitos. Os grupos sem diferença numérica são o melhor ponto de partida para a revisão por IA ou por pessoa.

## 5. Nomes confusos (flags)



In [90]:
# Abas de referência (mostrar para conferir)
un  = pd.read_excel(ARQUIVO, sheet_name='2. Unidades de Medida')
mo  = pd.read_excel(ARQUIVO, sheet_name='3. Mão de Obra')
display(un)
display(mo)

# Todas as unidades da base existem na aba de unidades?
print('Unidades da base fora da aba 2:', set(df['um']) - set(un.iloc[:, 0]))

,Sigla Unidade de Medida,Descrição
0,CM,Centímetro
1,CDM,Centímetro cúbico
2,ZCJ,Conjunto
3,TAG,Dia
4,H,Hora
5,L,Litro
6,MON,Mês
7,M,Metro
8,M3,Metro cúbico
9,M2,Metro quadrado


,Especificidades Mão de Obra,Descrição
0,Hora normal,"Hora de trabalho em jornada padrão, sem nenhum adicional aplicado — a hora ""..."
1,Hora extra Seg a Sab,"Hora trabalhada além da jornada normal em dia de semana ou sábado, sujeita a..."
2,Hora extra Dom/Feriado,"Hora trabalhada além da jornada normal em domingo ou feriado, geralmente com..."
3,Adicional noturno (hora),Acréscimo pago sobre a hora normal quando o trabalho ocorre no período notur...
4,Hora extra noturna Seg a Sab,Combinação de hora extra (dia de semana/sábado) com o adicional noturno — pa...
5,Hora extra noturna Dom/Feriado,Combinação de hora extra em domingo/feriado com o adicional noturno — o cená...
6,Adicional periculosidade (hora),Acréscimo pago por hora trabalhada em atividade classificada como perigosa (...
7,Adicional periculosidade s/ hora extra,"Mesmo adicional de periculosidade, mas aplicado especificamente sobre horas ..."
8,Adicional periculosidade (mensal),"Mesmo adicional de periculosidade, mas cobrado como valor fixo mensal em vez..."
9,Adicional insalubridade (hora),Acréscimo pago por hora quando a atividade expõe o trabalhador a agentes noc...


Unidades da base fora da aba 2: set()
